# 📚 Simple RAG Agent

## Learning Objectives
In this notebook, you will learn:
1. **Document Ingestion** — Load PDFs and create a vector store with embeddings
2. **Retriever as Tool** — Wrap a vector store retriever as a LangGraph tool
3. **Document Grading** — Use an LLM to assess whether retrieved documents are relevant
4. **Query Rewriting** — Transform user queries for improved retrieval
5. **Agentic RAG Graph** — Build a complete LangGraph workflow that retrieves, grades, and generates answers

## Prerequisites
- Completion of Chapters 01 (Foundations) and 02 (Core Capabilities)
- API keys configured in `.env` (OpenAI or Databricks)
- PDF document in `./docs/` directory

---

## 🔧 Part 1: Environment Setup

We begin by importing all required libraries and initializing our helper functions. All LLM and embedding initialization goes through the project's `helpers` factory — this ensures consistent configuration across notebooks.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Import Libraries and Helper Functions
# ============================================================================

import os, sys
import pprint
import warnings

warnings.filterwarnings("ignore")

from dotenv import load_dotenv
from typing import Annotated, Literal, Sequence, TypedDict
from pydantic import BaseModel, Field
from IPython.display import Image, display

# --- LangChain / LangGraph ---
from langchain_core.tools.retriever import create_retriever_tool
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langgraph.graph import END, StateGraph, START
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langsmith import Client

# --- Project helpers ---
sys.path.append(os.path.abspath(".."))
from helpers.utils import get_llm, get_databricks_embeddings

load_dotenv()

print("✅ All libraries imported successfully!")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: Create the Language Model
# ============================================================================

llm = get_llm()
# llm = get_llm(provider="openai", model="gpt-4-turbo")
# llm = get_llm(provider="groq")

---

## 📄 Part 2: Document Ingestion & Vector Store

In this section, we load a research paper (PDF), split it into chunks, embed them using a Databricks-hosted embedding model, and store them in a Chroma vector database. The retriever is then wrapped as a LangGraph-compatible **tool** that the agent can invoke.

In [ ]:
# ============================================================================
# DOCUMENT LOADING: Load and Split the PDF
# ============================================================================

file_path = './docs/human-agent-collab-problem-solving.pdf'
docs = PyPDFLoader(file_path).load_and_split()

print(f"📄 Loaded {len(docs)} document chunks from PDF")

In [ ]:
# ============================================================================
# DOCUMENT INSPECTION: Preview the Loaded Data
# ============================================================================

print(f"📋 Total chunks: {len(docs)}\n")
docs[0]

In [ ]:
# ============================================================================
# VECTOR STORE: Create Embeddings and Index Documents
# ============================================================================

# Initialize the embedding model
embeddings = get_databricks_embeddings("databricks-gte-large-en")

# Create vector store and retriever
vectorstore = Chroma.from_documents(
    documents=docs,
    collection_name="rag-chroma",
    embedding=embeddings,
)
retriever = vectorstore.as_retriever()

print("✅ Vector store created and retriever initialized!")

In [ ]:
# ============================================================================
# RETRIEVER TOOL: Wrap the Retriever as a LangGraph Tool
# ============================================================================

retriever_tool = create_retriever_tool(
    retriever,
    "retrieve_info_from_papers",
    "Search and return information about a paper discussing usage of LLMs for human collaborative problem solving.",
)

tools = [retriever_tool]

print(f"🔧 Created retriever tool: {retriever_tool.name}")

---

## 🧩 Part 3: Define Graph Components

Now we define the core building blocks of our agentic RAG graph:

1. **Agent State** — The shared state schema with message history
2. **Conditional Edge** — A document grader that routes to `generate` (if relevant) or `rewrite` (if not)
3. **Node Functions** — The `agent`, `rewrite`, and `generate` logic

### Key Concepts:
- **Document Grading**: An LLM assesses whether retrieved documents actually answer the question
- **Query Rewriting**: If documents aren't relevant, the agent reformulates the query and retries
- **RAG Prompt**: We pull a standard RAG prompt from LangSmith Hub (`rlm/rag-prompt`)

In [ ]:
# ============================================================================
# AGENT STATE: Define the Graph State Schema
# ============================================================================

class AgentState(TypedDict):
    # add_messages appends new messages instead of replacing
    messages: Annotated[Sequence[BaseMessage], add_messages]

In [ ]:
# ============================================================================
# CONDITIONAL EDGE: Document Relevance Grading
# ============================================================================

def grade_documents(state) -> Literal["generate", "rewrite"]:
    """
    Determines whether the retrieved documents are relevant to the question.
    Routes to 'generate' if relevant, 'rewrite' if not.
    """
    print("---CHECK RELEVANCE---")

    # Structured output schema for grading
    class grade(BaseModel):
        """Binary score for relevance check."""
        binary_score: str = Field(description="Relevance score 'yes' or 'no'")

    # LLM with structured output
    model = get_llm()
    llm_with_tool = model.with_structured_output(grade)

    # Grading prompt
    prompt = PromptTemplate(
        template="""You are a grader assessing relevance of a retrieved document to a user question. \n 
        Here is the retrieved document: \n\n {context} \n\n
        Here is the user question: {question} \n
        If the document contains keyword(s) or semantic meaning related to the user question, grade it as relevant. \n
        Give a binary score 'yes' or 'no' score to indicate whether the document is relevant to the question.""",
        input_variables=["context", "question"],
    )

    chain = prompt | llm_with_tool

    messages = state["messages"]
    last_message = messages[-1]
    question = messages[0].content
    docs = last_message.content

    scored_result = chain.invoke({"question": question, "context": docs})
    score = scored_result.binary_score

    if score == "yes":
        print("---DECISION: DOCS RELEVANT---")
        return "generate"
    else:
        print("---DECISION: DOCS NOT RELEVANT---")
        print(score)
        return "rewrite"

In [ ]:
# ============================================================================
# NODE FUNCTIONS: Agent, Rewrite, and Generate
# ============================================================================

def agent(state):
    """
    Invokes the agent model to decide whether to retrieve or end.
    The LLM is bound with the retriever tool so it can call it.
    """
    print("---CALL AGENT---")
    messages = state["messages"]
    model = get_llm()
    model = model.bind_tools(tools)
    response = model.invoke(messages)
    return {"messages": [response]}


def rewrite(state):
    """Transform the query to produce a better question for retrieval."""
    print("---TRANSFORM QUERY---")
    messages = state["messages"]
    question = messages[0].content

    msg = [
        HumanMessage(
            content=f""" \n 
    Look at the input and try to reason about the underlying semantic intent / meaning. \n 
    Here is the initial question:
    \n ------- \n
    {question} 
    \n ------- \n
    Formulate an improved question: """,
        )
    ]

    model = get_llm()
    response = model.invoke(msg)
    return {"messages": [response]}


def generate(state):
    """Generate an answer using retrieved documents as context."""
    print("---GENERATE---")
    messages = state["messages"]
    question = messages[0].content
    last_message = messages[-1]
    docs = last_message.content

    # Pull the RAG prompt from LangSmith Hub
    client = Client()
    prompt = client.pull_prompt("rlm/rag-prompt", dangerously_pull_public_prompt=True)

    llm = get_llm()
    rag_chain = prompt | llm | StrOutputParser()

    response = rag_chain.invoke({"context": docs, "question": question})
    return {"messages": [response]}

In [ ]:
# ============================================================================
# PROMPT INSPECTION: View the RAG Prompt Template
# ============================================================================

print("*" * 20 + " Prompt[rlm/rag-prompt] " + "*" * 20)
client = Client()
client.pull_prompt("rlm/rag-prompt", dangerously_pull_public_prompt=True).pretty_print()

---

## 🏗️ Part 4: Assemble & Run the Graph

We connect all the components into a LangGraph `StateGraph`. The workflow follows this pattern:

```
START → Agent → [retrieve or END]
                  ↓
              Retrieve → Grade Documents → [generate or rewrite]
                                              ↓           ↓
                                            END        Agent (retry)
```

The agent autonomously decides whether retrieval is needed, grades the results, and either generates an answer or rewrites the query for another attempt.

In [ ]:
# ============================================================================
# GRAPH CONSTRUCTION: Assemble the Agentic RAG Workflow
# ============================================================================

workflow = StateGraph(AgentState)

# --- Add nodes ---
workflow.add_node("agent", agent)
workflow.add_node("retrieve", ToolNode(tools))
workflow.add_node("rewrite", rewrite)
workflow.add_node("generate", generate)

# --- Add edges ---
workflow.add_edge(START, "agent")

# Agent decides: retrieve or end
workflow.add_conditional_edges(
    "agent",
    tools_condition,
    {"tools": "retrieve", END: END},
)

# After retrieval: grade documents → generate or rewrite
workflow.add_conditional_edges("retrieve", grade_documents)
workflow.add_edge("generate", END)
workflow.add_edge("rewrite", "agent")

# Compile the graph
graph = workflow.compile()

print("✅ Graph compiled successfully!")

In [ ]:
# ============================================================================
# GRAPH VISUALIZATION: Display the Agent Workflow
# ============================================================================

try:
    display(Image(graph.get_graph(xray=True).draw_mermaid_png()))
except Exception:
    print("⚠️ Graph visualization requires additional dependencies")

In [ ]:
# ============================================================================
# EXECUTION: Run the RAG Agent
# ============================================================================

inputs = {
    "messages": [
        ("user", "In this paper how do the authors set up the collaboration between the human and the LLMs?"),
    ]
}

for output in graph.stream(inputs):
    for key, value in output.items():
        pprint.pprint(f"Output from node '{key}':")
        pprint.pprint("---")
        pprint.pprint(value, indent=2, width=80, depth=None)
    pprint.pprint("\n---\n")

---

## 📝 Summary

In this notebook, we built a **Simple RAG Agent** using LangGraph that:

### 1. Document Ingestion
- Loaded a PDF paper and split it into chunks
- Created a vector store using Chroma with Databricks GTE embeddings

### 2. Agentic Retrieval
- Wrapped the vector store retriever as a **LangGraph tool**
- The agent autonomously decides when to use the retriever

### 3. Document Grading & Query Rewriting
- **Grading**: An LLM evaluates whether retrieved documents are relevant to the question
- **Rewriting**: If documents aren't relevant, the query is reformulated and retrieval retried

### 4. Graph Architecture
The workflow follows: `Agent → Retrieve → Grade → Generate/Rewrite` with conditional edges routing based on document relevance and a rewrite loop for self-correction.

### Next Steps
- **02_Simple_RAG_Agent_Databricks** — Same pattern using Databricks Vector Search
- **03_Advanced_RAG_Agent** — Advanced RAG with query enhancement, topic validation, and optimization loops
- **04_RAG_as_Tool_in_Agents** — RAG wrapped as a tool the LLM decides when to invoke